# Transformer 아키텍처 완전 해부
## 실습 코드 1: Transformer Encoder Block (PyTorch)

---

이 노트북은 자연어 처리(NLP)를 혁신한 **Transformer** 모델의 핵심 구성 단위인  
**Encoder Block**을 처음부터 단계별로 구현하고 이해하기 위한 실험 노트입니다.

---

### 🎯 학습 목표
1. 입력 텐서(단어 벡터)가 **Q / K / V**로 변환되는 과정을 shape와 함께 추적한다
2. **Scaled Dot-Product Attention** 수식을 코드로 직접 구현한다
3. **Multi-Head Attention**이 "여러 관점"을 동시에 학습하는 원리를 이해한다
4. **Pre-LayerNorm + 잔차 연결(Residual Connection)** 의 역할을 이해한다
5. 완성된 Encoder Block이 입력과 **동일한 shape**를 유지함을 확인한다

---

### 📚 사전 지식 (없어도 됩니다 — 주석이 설명해 드립니다)
- Python 기초 문법
- 행렬(m×n 배열) 의 개념
- PyTorch 텐서 기초 (선택)

---

### 🗺️ 전체 흐름 미리보기
```
입력 문장  →  [토큰 임베딩]  →  shape: (batch, seq_len, d_model)
                                           │
                          ┌────────────────┤
                          │                │
                  LayerNorm              (x 보존)
                          │                │
               Multi-Head Attention        │
                          │                │
                    Dropout + ────────────→+ (잔차 연결)
                                           │
                          ┌────────────────┤
                          │                │
                  LayerNorm              (x 보존)
                          │                │
               Feed-Forward Network        │
                          │                │
                    Dropout + ────────────→+ (잔차 연결)
                                           │
                                        출력 (입력과 동일한 shape)
```

---

### 📂 노트북 구성
| 섹션 | 내용 |
|------|------|
| Step 0 | 환경 설정 & 하이퍼파라미터 |
| Step 1 | 입력 텐서 이해 |
| Step 2 | Self-Attention 직접 구현 (Q, K, V) |
| Step 3 | Multi-Head Attention 직접 구현 |
| Step 4 | Feed-Forward Network |
| Step 5 | LayerNorm & Residual Connection |
| Step 6 | 완성: Transformer Encoder Block |
| Step 7 | 실험 & 시각화 |

## Step 0-A: 라이브러리 임포트

> 이 노트북에서 사용할 도구들을 불러옵니다.

In [ ]:
# ──────────────────────────────────────────────────────
# 라이브러리 임포트
# ──────────────────────────────────────────────────────

import torch                        # PyTorch: 딥러닝 프레임워크
import torch.nn as nn               # nn: 레이어, 손실함수 등을 모아둔 모듈
import torch.nn.functional as F     # F: softmax, relu 같은 함수들
import math                          # 수학 함수 (sqrt 등)
import matplotlib.pyplot as plt      # 시각화 (그래프, 히트맵)
import matplotlib
import numpy as np                   # 수치 계산

# ── 한글 폰트 설정 (matplotlib) ──
# 시각화에서 한글이 깨지지 않도록 설정합니다.
# Colab이나 로컬 환경에 따라 폰트 경로가 다를 수 있습니다.
try:
    matplotlib.rcParams['font.family'] = 'NanumGothic'
except:
    pass  # 폰트 없어도 실습에는 문제 없습니다

matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# ── 재현성을 위한 시드 고정 ──
# 시드(seed)를 고정하면 매번 실행해도 동일한 난수가 생성되어
# 실험 결과가 일관되게 유지됩니다.
torch.manual_seed(42)
np.random.seed(42)

print("✅ 라이브러리 임포트 완료!")
print(f"PyTorch 버전: {torch.__version__}")

## Step 0-B: 하이퍼파라미터 설정

> **하이퍼파라미터** = 모델 학습 전에 사람이 직접 정하는 설정값.  
> 이 값들이 모델의 크기와 능력을 결정합니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 하이퍼파라미터 설정
# ──────────────────────────────────────────────────────────────────

# ── 모델 차원 ──────────────────────────────────────────────────────
d_model = 512
# d_model: 각 토큰(단어)을 표현하는 벡터의 크기
# 예) "고양이"라는 단어를 512개의 숫자로 표현합니다.
# 클수록 표현력이 높아지지만, 계산량도 증가합니다.
# 참고) BERT-base=768, GPT-2=768, GPT-3=12288

# ── 어텐션 헤드 수 ─────────────────────────────────────────────────
num_heads = 8
# num_heads: 몇 개의 "관점"에서 동시에 어텐션을 수행할지
# 각 헤드는 d_model/num_heads = 512/8 = 64 차원을 담당합니다.
# 여러 헤드가 각자 다른 관계(문법, 의미, 위치 등)를 포착합니다.

d_k = d_model // num_heads  # 헤드당 Query/Key 차원 = 512 // 8 = 64
d_v = d_model // num_heads  # 헤드당 Value 차원 = 64

# ── Feed-Forward 내부 차원 ─────────────────────────────────────────
d_ff = 2048
# d_ff: FFN(Feed-Forward Network) 내부 은닉층의 크기
# 관행적으로 d_model의 4배를 씁니다: 512 × 4 = 2048
# 어텐션이 "어떤 단어를 볼지" 결정한다면,
# FFN은 "그 정보로 무엇을 할지" 비선형 변환하는 단계입니다.

# ── 드롭아웃 ───────────────────────────────────────────────────────
dropout = 0.1
# dropout: 학습 중 일부 뉴런을 무작위로 끄는 기법 (과적합 방지)
# 0.1 = 10%의 활성화 값을 0으로 만들겠다는 뜻입니다.
# 학습(train) 모드에서만 동작하고, 추론(eval) 모드에서는 비활성화됩니다.

# ── 배치 및 시퀀스 ────────────────────────────────────────────────
batch_size = 2
# batch_size: 한 번에 처리할 문장의 수
# 예) 2개의 문장을 GPU에 동시에 올려서 병렬 처리

seq_len = 10
# seq_len: 각 문장의 최대 토큰(단어) 수
# 예) "나는 오늘 학교에 갔다" = 4개 토큰 → seq_len=10이면 패딩 필요

# ── 요약 출력 ─────────────────────────────────────────────────────
print("=" * 55)
print(" 하이퍼파라미터 설정 요약")
print("=" * 55)
print(f"  d_model   (토큰 임베딩 차원) : {d_model}")
print(f"  num_heads (어텐션 헤드 수)   : {num_heads}")
print(f"  d_k = d_v (헤드당 차원)      : {d_k}  ({d_model} / {num_heads})")
print(f"  d_ff      (FFN 내부 차원)    : {d_ff}  ({d_model} × 4)")
print(f"  dropout                       : {dropout}")
print(f"  batch_size                    : {batch_size}")
print(f"  seq_len                       : {seq_len}")

## Step 1: 입력 텐서 이해하기

Transformer의 입력은 **3차원 텐서**입니다:

```
shape = (batch_size, seq_len, d_model)
          ↑           ↑        ↑
       문장 수     토큰 수  토큰 벡터 크기
```

#### 직관적 이해 — 박스로 시각화하면:
```
batch 0: [  토큰0  |  토큰1  |  토큰2  | ... |  토큰9  ]
           [512개]   [512개]   [512개]         [512개]
batch 1: [  토큰0  |  토큰1  |  토큰2  | ... |  토큰9  ]
```

> 💡 실제 모델에서는 "나는", "오늘" 같은 단어를 정수 인덱스로 바꾼 뒤  
> `nn.Embedding`을 통해 512차원 벡터로 변환합니다.  
> 여기서는 **이미 변환된 상태**를 가정하고 `torch.randn`으로 흉내 냅니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 입력 텐서 생성 및 shape 확인
# ──────────────────────────────────────────────────────────────────

# torch.randn: 평균 0, 표준편차 1의 정규분포에서 무작위 숫자를 뽑아 텐서를 만듭니다.
# 실제로는 단어 임베딩 값이 여기에 들어옵니다.
x = torch.randn(batch_size, seq_len, d_model)

print(f"x.shape = {x.shape}")
print()
print("각 차원의 의미:")
print(f"  x.shape[0] = {x.shape[0]}   → {batch_size}개의 문장을 동시에 처리 (batch)")
print(f"  x.shape[1] = {x.shape[1]}  → 각 문장은 {seq_len}개의 토큰으로 구성")
print(f"  x.shape[2] = {x.shape[2]} → 각 토큰은 {d_model}차원 벡터로 표현")
print()
print("── 첫 번째 문장(batch 0)의 첫 번째 토큰 벡터 (처음 8개 차원) ──")
print(x[0, 0, :8].tolist())
print("  ↑ 이 512개의 숫자가 하나의 단어(토큰)를 표현합니다.")
print()

# 전체 입력에 담긴 숫자의 총 개수를 계산해 봅니다.
total_numbers = batch_size * seq_len * d_model
print(f"전체 원소 수: {batch_size} × {seq_len} × {d_model} = {total_numbers:,}개")

## Step 2: Self-Attention 직접 구현하기

### 📖 Self-Attention이란?

Self-Attention은 **"각 단어가 문장 안의 다른 단어들을 얼마나 참조할지" 결정하는 메커니즘**입니다.

#### 비유로 이해하기 — 도서관 검색 시스템

| 이름 | 역할 | 비유 |
|------|------|------|
| **Query (Q)** | 내가 찾는 것 | 검색어: "파이썬 프로그래밍" |
| **Key (K)** | 각 책의 라벨 | 책 제목: "파이썬", "자바", "C++" |
| **Value (V)** | 실제 책 내용 | 책 본문 |

→ Q와 K를 비교해서 **유사도 점수**를 매기고,  
→ 그 점수를 가중치로 삼아 V를 **가중합**합니다.

#### 수식

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) \cdot V$$

- $QK^T$: Q와 K의 내적 → "얼마나 유사한가?" 점수
- $/ \sqrt{d_k}$: 스케일링 → 차원이 크면 값이 커지므로 나눠서 조정
- $\text{softmax}$: 점수를 **확률(합=1)** 로 변환
- $\cdot V$: 확률로 V를 가중합

#### "Self"인 이유

Q, K, V를 **모두 같은 입력 x**에서 만들기 때문입니다.  
(Decoder의 Cross-Attention은 K, V가 Encoder 출력에서 옵니다.)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 2-1: Q, K, V 계산
# ──────────────────────────────────────────────────────────────────

# 가중치 행렬 정의
# nn.Linear(in, out): in차원 입력을 out차원으로 선형 변환하는 레이어
# 행렬로 표현하면 W: (d_model × d_model)
# 실제 학습에서는 이 가중치가 역전파(backprop)로 업데이트됩니다.
W_Q = nn.Linear(d_model, d_model, bias=False)  # Query 투영
W_K = nn.Linear(d_model, d_model, bias=False)  # Key 투영
W_V = nn.Linear(d_model, d_model, bias=False)  # Value 투영

# Q, K, V 계산
# 모두 같은 입력 x를 사용합니다 → "Self"-Attention!
# W_Q(x): x와 W_Q를 행렬곱 → (batch, seq_len, d_model)
Q = W_Q(x)   # shape: (2, 10, 512)
K = W_K(x)   # shape: (2, 10, 512)
V = W_V(x)   # shape: (2, 10, 512)

print("Q, K, V 계산 결과:")
print(f"  입력  x.shape = {x.shape}")
print(f"  Q.shape       = {Q.shape}  ← x × W_Q")
print(f"  K.shape       = {K.shape}  ← x × W_K")
print(f"  V.shape       = {V.shape}  ← x × W_V")
print()
print("💡 Q, K, V는 입력 x와 shape가 같습니다.")
print("   각각 다른 선형 변환(가중치 행렬)을 거쳐 서로 다른 '역할'을 맡습니다.")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 2-2: Attention Score 계산
# ──────────────────────────────────────────────────────────────────
#
# scores = Q × K^T / sqrt(d_k)
#
# 결과: (batch, seq_len, seq_len) 크기의 행렬
# 각 (i, j) 위치의 값 = i번째 토큰이 j번째 토큰에 얼마나 집중하는지
#
# 예시 (seq_len=4, 문장: "나는 오늘 학교에 갔다"):
#
#            나는   오늘  학교에  갔다
#   나는  [ 0.8   0.1    0.2    0.3 ]  ← "나는"이 각 단어를 얼마나 참조하는지
#   오늘  [ 0.2   0.7    0.2    0.1 ]
#   학교에 [ 0.1   0.1    0.6    0.3 ]
#   갔다  [ 0.2   0.1    0.3    0.7 ]
#          ↑ 각 열은 해당 단어가 얼마나 참조받는지를 나타냅니다

# ① Q × K^T
# K.transpose(-2, -1): K의 마지막 두 차원을 바꿉니다
#   K:   (2, 10, 512) → K^T: (2, 512, 10)
# matmul 결과: (2, 10, 10)
scores = torch.matmul(Q, K.transpose(-2, -1))
print("① Q × K^T 계산")
print(f"   Q.shape        = {Q.shape}")
print(f"   K^T.shape      = {K.transpose(-2,-1).shape}")
print(f"   scores.shape   = {scores.shape}  ← (batch, seq_len, seq_len)")
print()

# ② 스케일링: / sqrt(d_k)
# 왜 나누나요?
#   d_model(=512)이 커질수록 내적 값의 분산이 커집니다.
#   값이 너무 크면 softmax가 극단적(거의 0 또는 1)이 되고
#   그라디언트가 거의 0이 되어 학습이 느려집니다.
#   sqrt(d_k)로 나누면 분산을 1로 정규화할 수 있습니다.
scale = math.sqrt(d_k)   # sqrt(64) = 8.0
scores_scaled = scores / scale
print(f"② 스케일링: / sqrt(d_k) = / {scale:.1f}")
print(f"   스케일링 전 분산: {scores.var().item():.4f}")
print(f"   스케일링 후 분산: {scores_scaled.var().item():.4f}  (더 작아졌습니다)")
print()

# ③ Softmax: 각 행의 합이 1이 되도록 정규화
# dim=-1: 마지막 차원(seq_len 방향)으로 softmax 적용
#   즉, 각 토큰(행)이 다른 모든 토큰에 주는 가중치의 합 = 1
attn_weights = F.softmax(scores_scaled, dim=-1)
print(f"③ Softmax 후 attention weights")
print(f"   shape = {attn_weights.shape}")
print(f"   첫 번째 배치, 첫 번째 토큰의 가중치 (처음 5개):")
print(f"   {attn_weights[0, 0, :5].detach().tolist()}")
print(f"   → 합계: {attn_weights[0, 0].sum().item():.6f}  (반드시 1.0이어야 합니다)")
print()

# ④ Value에 어텐션 가중치 적용
# attn_weights: (2, 10, 10)  V: (2, 10, 512)
# 결과:         (2, 10, 512)
#
# 각 토큰의 새 표현 = 모든 토큰의 Value를 어텐션 가중치로 가중합한 것
attn_output = torch.matmul(attn_weights, V)
print(f"④ Attention output")
print(f"   attn_weights.shape × V.shape → attn_output.shape")
print(f"   {attn_weights.shape}    ×  {V.shape}  → {attn_output.shape}")
print()
print("✅ Single-Head Self-Attention 완성!")
print("   각 토큰의 새 표현에 문장 전체의 문맥이 녹아들었습니다.")

## Step 3: Multi-Head Attention

### 왜 여러 헤드가 필요한가?

"나는 어제 **은행**에 갔다"에서 **은행**의 의미를 파악할 때:

- **헤드 1**: 문법 관계 포착 → "갔다"와 연결
- **헤드 2**: 의미 관계 포착 → "돈", "금융"과 연결
- **헤드 3**: 위치 정보 포착 → 앞뒤 몇 번째 토큰인지
- **헤드 4~8**: 기타 다양한 패턴...

하나의 헤드로는 이 모든 관계를 동시에 포착하기 어렵습니다!

### 구현 원리 — Shape 변화 추적

```
(batch, seq_len, d_model)
→ Q/K/V 투영: (batch, seq_len, d_model)
→ 헤드 분리:  (batch, seq_len, num_heads, d_k)
→ 전치:       (batch, num_heads, seq_len, d_k)
→ 각 헤드별 Attention: (batch, num_heads, seq_len, d_k)
→ 헤드 합치기: (batch, seq_len, d_model)
→ 최종 투영:   (batch, seq_len, d_model)
```

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 3-1: Scaled Dot-Product Attention 함수
# (Single-Head 또는 Multi-Head 공통으로 사용)
# ──────────────────────────────────────────────────────────────────

def scaled_dot_product_attention(q, k, v, mask=None):
    """
    Scaled Dot-Product Attention

    Args:
        q:    Query 텐서,  shape (batch, heads, seq_len, d_k)
        k:    Key 텐서,    shape (batch, heads, seq_len, d_k)
        v:    Value 텐서,  shape (batch, heads, seq_len, d_v)
        mask: 어텐션 마스크 (선택), shape (seq_len, seq_len)
              마스크가 0인 위치는 어텐션이 0이 됩니다.

    Returns:
        output:  어텐션 적용 출력,  shape (batch, heads, seq_len, d_v)
        weights: 어텐션 가중치,     shape (batch, heads, seq_len, seq_len)
    """
    d_k = q.size(-1)  # 마지막 차원 크기 = d_k

    # ① QK^T / sqrt(d_k)
    # q:   (batch, heads, seq, d_k)
    # k^T: (batch, heads, d_k, seq)
    # →    (batch, heads, seq, seq)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)

    # ② 마스크 적용 (있을 때만)
    # 마스크가 있는 위치에 -inf를 넣으면
    # softmax 후 해당 위치의 가중치가 0이 됩니다.
    # (e^(-inf) ≈ 0)
    if mask is not None:
        scores = scores + mask  # -inf가 더해진 위치는 softmax 후 0

    # ③ Softmax → 확률로 변환
    weights = F.softmax(scores, dim=-1)

    # ④ V와 가중합
    output = torch.matmul(weights, v)

    return output, weights

print("scaled_dot_product_attention 함수 정의 완료!")
print()
print("입력 → 출력 shape 요약:")
print("  q, k, v: (batch, heads, seq_len, d_k)")
print("  output:  (batch, heads, seq_len, d_k)")
print("  weights: (batch, heads, seq_len, seq_len)")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 3-2: Multi-Head Attention 직접 구현
# ──────────────────────────────────────────────────────────────────

class ManualMultiHeadAttention(nn.Module):
    """
    Multi-Head Attention을 단계별로 직접 구현한 클래스.
    내부 shape 변화를 눈으로 확인할 수 있습니다.
    (nn.MultiheadAttention과 동일한 동작)
    """

    def __init__(self, d_model, num_heads):
        super().__init__()
        # 전제 조건: d_model은 num_heads의 배수여야 합니다.
        assert d_model % num_heads == 0, (
            f"d_model({d_model})은 num_heads({num_heads})로 나누어 떨어져야 합니다!"
        )

        self.d_model    = d_model
        self.num_heads  = num_heads
        self.d_k        = d_model // num_heads   # 헤드당 차원

        # ─ 선형 투영 레이어 4개 ─
        # Q, K, V를 d_model 공간으로 투영한 뒤, 헤드로 분리합니다.
        self.W_q = nn.Linear(d_model, d_model)  # Query 투영
        self.W_k = nn.Linear(d_model, d_model)  # Key 투영
        self.W_v = nn.Linear(d_model, d_model)  # Value 투영
        self.W_o = nn.Linear(d_model, d_model)  # 출력 투영 (헤드 합친 후)

    def split_heads(self, x, label=""):
        """
        (batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k)

        d_model을 num_heads × d_k로 쪼개어 각 헤드에 할당합니다.
        """
        batch, seq_len, _ = x.shape
        # 마지막 차원을 (num_heads, d_k) 로 분리
        x = x.view(batch, seq_len, self.num_heads, self.d_k)
        # heads 차원을 앞으로: (batch, seq, heads, d_k) → (batch, heads, seq, d_k)
        x = x.transpose(1, 2)
        if label:
            print(f"  split_heads({label}): (batch, seq, d_model) "
                  f"→ {x.shape}")
        return x

    def combine_heads(self, x):
        """
        (batch, num_heads, seq_len, d_k) → (batch, seq_len, d_model)

        분리된 헤드들을 다시 d_model 차원으로 합칩니다.
        """
        batch, heads, seq_len, d_k = x.shape
        # (batch, heads, seq, d_k) → (batch, seq, heads, d_k)
        x = x.transpose(1, 2)
        # (batch, seq, heads * d_k) = (batch, seq, d_model)
        # contiguous(): transpose 후 메모리 배치를 연속으로 만들어야 view 가능
        x = x.contiguous().view(batch, seq_len, self.d_model)
        return x

    def forward(self, x, mask=None, verbose=False):
        """
        Args:
            x:       입력 텐서, shape (batch, seq_len, d_model)
            mask:    어텐션 마스크 (선택)
            verbose: True이면 각 단계별 shape를 출력
        Returns:
            output:       어텐션 출력, shape (batch, seq_len, d_model)
            attn_weights: 어텐션 가중치, shape (batch, heads, seq, seq)
        """
        if verbose: print(f"[MHA 시작] 입력 x.shape = {x.shape}")

        # ① Q, K, V 선형 투영
        Q = self.W_q(x)   # (batch, seq, d_model)
        K = self.W_k(x)   # (batch, seq, d_model)
        V = self.W_v(x)   # (batch, seq, d_model)
        if verbose:
            print(f"  ① 선형 투영 후: Q/K/V.shape = {Q.shape}")

        # ② 헤드 분리: (batch, seq, d_model) → (batch, heads, seq, d_k)
        if verbose: print("  ② 헤드 분리:")
        Q = self.split_heads(Q, "Q" if verbose else "")
        K = self.split_heads(K, "K" if verbose else "")
        V = self.split_heads(V, "V" if verbose else "")

        # ③ 각 헤드에서 Scaled Dot-Product Attention 수행
        # PyTorch가 batch/heads 차원을 자동으로 병렬 처리합니다.
        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        if verbose:
            print(f"  ③ 어텐션 후: attn_out.shape    = {attn_out.shape}")
            print(f"               attn_weights.shape = {attn_weights.shape}")

        # ④ 헤드 합치기: (batch, heads, seq, d_k) → (batch, seq, d_model)
        attn_out = self.combine_heads(attn_out)
        if verbose: print(f"  ④ 헤드 합친 후: {attn_out.shape}")

        # ⑤ 최종 선형 투영
        output = self.W_o(attn_out)   # (batch, seq, d_model)
        if verbose: print(f"  ⑤ 출력 투영 후: {output.shape}  [최종 출력]")

        return output, attn_weights


# ── 테스트 (verbose=True로 각 단계 확인) ──
print("=" * 55)
print(" ManualMultiHeadAttention 단계별 shape 추적")
print("=" * 55)
mha_manual = ManualMultiHeadAttention(d_model=512, num_heads=8)
out_m, w_m = mha_manual(x, verbose=True)
print()
print(f"최종 확인")
print(f"  입력  x.shape      = {x.shape}")
print(f"  출력  out_m.shape  = {out_m.shape}")
print(f"  가중치 w_m.shape   = {w_m.shape}")
print()
print(f"✅ {num_heads}개의 헤드가 각각 {d_k}차원({d_model}/{num_heads})을 담당하며 "
      f"병렬로 어텐션을 수행했습니다!")

## Step 4: Feed-Forward Network (FFN)

어텐션이 **"어떤 단어를 볼지"** 결정했다면,  
FFN은 **"그 정보로 무엇을 할지"** 처리하는 단계입니다.

### 구조
```
Linear(d_model → d_ff)  →  GELU  →  Linear(d_ff → d_model)
     512 → 2048                         2048 → 512
```

### 특징
- 각 토큰 위치에서 **독립적으로** 동일한 FFN이 적용됩니다 (위치 무관)
- 어텐션이 토큰들 간 정보를 교환했다면, FFN은 각 위치에서 정보를 정제
- **GELU** 활성화 함수 사용 (ReLU의 부드러운 버전)

### GELU vs ReLU
```
ReLU(x) = max(0, x)      → x < 0이면 무조건 0
GELU(x) ≈ x × Φ(x)       → 음수도 일부 통과시키는 부드러운 곡선
```
실제 Transformer 모델에서 GELU가 더 좋은 성능을 보입니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 4: Feed-Forward Network 구현 및 시각화
# ──────────────────────────────────────────────────────────────────

class FeedForwardNetwork(nn.Module):
    """
    Position-wise Feed-Forward Network

    구조: Linear(d_model→d_ff) → GELU → Dropout → Linear(d_ff→d_model) → Dropout

    Args:
        d_model (int): 입력/출력 차원 (기본값: 512)
        d_ff    (int): 내부 은닉층 차원 (기본값: 2048)
        dropout (float): 드롭아웃 비율 (기본값: 0.1)
    """

    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),   # 확장: 512 → 2048
            nn.GELU(),                   # 비선형 활성화 함수
            nn.Dropout(dropout),         # 학습 중 과적합 방지
            nn.Linear(d_ff, d_model),   # 축소: 2048 → 512
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        # 각 위치(seq_len)에서 동일한 FFN이 독립적으로 적용됩니다.
        # → 내부에서 (batch × seq_len, d_model) 처럼 동작하는 효과
        return self.net(x)  # shape 유지: (batch, seq_len, d_model)


# ── 테스트 ──
ffn = FeedForwardNetwork(d_model=512, d_ff=2048, dropout=0.1)
ffn_out = ffn(x)
print("Feed-Forward Network 테스트:")
print(f"  입력  x.shape       = {x.shape}")
print(f"  출력  ffn_out.shape = {ffn_out.shape}")
print()
print("내부 shape 변화:")
print(f"  ({batch_size},{seq_len},{d_model}) → Linear → ({batch_size},{seq_len},{d_ff}) "
      f"→ GELU → Linear → ({batch_size},{seq_len},{d_model})")
print()

# ── GELU vs ReLU 시각화 ──
x_vis = torch.linspace(-3, 3, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_vis.numpy(), F.gelu(x_vis).numpy(),  label="GELU", color="steelblue", linewidth=2.5)
ax.plot(x_vis.numpy(), F.relu(x_vis).numpy(),  label="ReLU", color="tomato",    linewidth=2,
        linestyle="--")
ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
ax.axvline(0, color="gray", linewidth=0.8, linestyle=":")
ax.set_title("GELU vs ReLU 활성화 함수", fontsize=13)
ax.set_xlabel("입력 x")
ax.set_ylabel("출력")
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("💡 GELU는 음수 영역에서도 아주 작은 기울기를 유지해 ReLU보다 부드럽습니다.")

## Step 5: LayerNorm & Residual Connection

### 5-A. Layer Normalization

딥러닝에서 레이어가 깊어질수록 값이 폭발하거나 사라지는 문제가 생깁니다.  
**Layer Normalization**은 각 토큰 벡터를 정규화해 학습을 안정화합니다.

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sigma + \epsilon} + \beta$$

- $\mu$: 해당 토큰 벡터의 평균  
- $\sigma$: 해당 토큰 벡터의 표준편차  
- $\gamma, \beta$: 학습 가능한 파라미터 (초기값: $\gamma=1$, $\beta=0$)

### 5-B. Residual Connection (잔차 연결)

```
output = x + SubLayer(x)   ← 입력 x를 그대로 더합니다
```

**역할:**
1. 그라디언트가 직접 흐르는 "고속도로"를 제공 → 기울기 소실 문제 해결
2. 레이어는 전체 함수가 아닌 **"보정값(잔차)"** 만 학습하면 됩니다

> 📝 **비유:** 학생이 시험 답안지를 처음부터 다시 쓰는 대신,  
> 원래 답안지에 **빨간 펜으로 수정 사항만** 표시하는 것!

### Pre-LN vs Post-LN

| 방식 | 수식 | 사용처 |
|------|------|--------|
| **Post-LN** (원래 논문) | `LayerNorm(x + SubLayer(x))` | 원래 Transformer |
| **Pre-LN** (현재 주류) | `x + SubLayer(LayerNorm(x))` | GPT, BERT 등 |

Pre-LN이 학습 초기에 더 안정적이어서 현재 더 많이 사용됩니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 5-A: Layer Normalization 동작 확인
# ──────────────────────────────────────────────────────────────────

print("=" * 50)
print(" LayerNorm 동작 시연")
print("=" * 50)

# 간단한 예시: 값 차이가 큰 벡터
v = torch.tensor([[1.0, 100.0, 3.0, 50.0, 5.0]])  # shape: (1, 5)
ln5 = nn.LayerNorm(5)  # 5차원에 대해 정규화

print(f"원본 벡터: {v.tolist()[0]}")
print(f"  평균:       {v.mean().item():.2f}")
print(f"  표준편차:   {v.std().item():.2f}")
print()

normed5 = ln5(v)
print(f"LayerNorm 후: {[round(x,3) for x in normed5.detach().tolist()[0]]}")
print(f"  평균 (≈ 0): {normed5.mean().item():.6f}")
print(f"  표준편차 (≈ 1): {normed5.std(unbiased=False).item():.6f}")
print()
print("💡 100이나 50처럼 큰 값들이 -1 ~ 1 근방으로 정규화되었습니다.")
print("   이렇게 하면 어떤 레이어든 비슷한 규모의 값을 입력받게 됩니다.")
print()

# ──────────────────────────────────────────────────────────────────
# Step 5-B: Residual Connection 시연
# ──────────────────────────────────────────────────────────────────

print("=" * 50)
print(" Residual Connection 시연 (Pre-LN 방식)")
print("=" * 50)

# 작은 크기로 직관적 확인
x_demo = torch.randn(1, 4, 16)   # (batch=1, seq=4, d=16)
ln16   = nn.LayerNorm(16)
sub    = nn.Linear(16, 16)       # 임의의 서브레이어

# Pre-LN Residual:
#   1) 먼저 LayerNorm
#   2) 서브레이어 통과
#   3) 원래 x에 더하기
normed_demo = ln16(x_demo)          # LayerNorm 적용
sub_out     = sub(normed_demo)       # 서브레이어 적용
output_demo = x_demo + sub_out      # 잔차 연결!

print(f"입력 x_demo:       {x_demo.shape}")
print(f"LayerNorm 후:      {normed_demo.shape}")
print(f"서브레이어 후:      {sub_out.shape}")
print(f"잔차 연결 후 출력: {output_demo.shape}")
print()
print("✅ 잔차 연결은 shape를 바꾸지 않습니다!")
print("   입력과 서브레이어 출력의 shape가 같아야 더할 수 있습니다.")

## Step 6: 완성 — Transformer Encoder Block

지금까지 만든 **모든 구성 요소를 조립**합니다!

```
x (입력)
│
├── x ────────────────────────────────────────────────────┐
│                                                          │ (잔차)
└── LayerNorm(x) → MultiHeadAttention → Dropout ─────────→ + → x'
                                                           │
├── x' ───────────────────────────────────────────────────┐
│                                                          │ (잔차)
└── LayerNorm(x') → FFN → Dropout ──────────────────────→ + → 출력
```

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 6: Transformer Encoder Block (완성본)
# ──────────────────────────────────────────────────────────────────

class TransformerEncoderBlock(nn.Module):
    """
    Transformer Encoder Block (Pre-LN 방식)

    구성 요소:
      1. Multi-Head Self-Attention  (+ Pre-LayerNorm + Residual)
      2. Feed-Forward Network       (+ Pre-LayerNorm + Residual)

    핵심 특성:
      - 입력과 출력의 shape가 동일합니다: (batch, seq_len, d_model)
      - 따라서 여러 블록을 순서대로 쌓을 수 있습니다. (BERT: 12개, GPT-3: 96개)

    Args:
        d_model   (int):   토큰 임베딩 차원. 기본값: 512
        num_heads (int):   어텐션 헤드 수.   기본값: 8
        d_ff      (int):   FFN 내부 차원.    기본값: 2048
        dropout   (float): 드롭아웃 비율.    기본값: 0.1
    """

    def __init__(self, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()

        # ─ Multi-Head Attention ─────────────────────────────────────
        # nn.MultiheadAttention: PyTorch 내장 구현 (Step 3의 Manual 버전과 동일)
        # batch_first=True: 입력 shape을 (batch, seq, d_model)로 받겠다는 설정
        #   (기본값은 (seq, batch, d_model)이지만 직관적이지 않으므로 True로 설정)
        self.mha = nn.MultiheadAttention(
            embed_dim   = d_model,    # 임베딩 차원
            num_heads   = num_heads,  # 헤드 수 (d_model/num_heads 가 헤드당 차원)
            dropout     = dropout,    # 어텐션 가중치에 적용할 드롭아웃
            batch_first = True        # (batch, seq, d_model) 형식 사용
        )

        # ─ Feed-Forward Network ──────────────────────────────────────
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),   # 확장: 512 → 2048
            nn.GELU(),                   # 활성화 함수 (비선형성)
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),   # 축소: 2048 → 512
            nn.Dropout(dropout),
        )

        # ─ Layer Normalization (2개) ─────────────────────────────────
        # ln1: MHA 전에 적용
        # ln2: FFN 전에 적용
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # ─ Dropout ───────────────────────────────────────────────────
        # MHA와 FFN 출력에 각각 한 번 더 드롭아웃 적용
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Args:
            x    (Tensor): 입력 텐서, shape (batch_size, seq_len, d_model)
            mask (Tensor): 어텐션 마스크 (선택사항),
                           shape (seq_len, seq_len)
                           마스크된 위치는 -inf (어텐션 0 처리)
        Returns:
            Tensor: 출력 텐서, shape (batch_size, seq_len, d_model)
                    ← 입력과 반드시 동일한 shape!
        """

        # ════════════════════════════════════════════════
        # 서브레이어 1: Multi-Head Self-Attention
        # ════════════════════════════════════════════════

        # Pre-LN: 어텐션 연산 전에 LayerNorm 적용
        normed = self.ln1(x)
        # normed shape: (batch, seq_len, d_model) — shape 불변

        # Self-Attention 수행
        # query=normed, key=normed, value=normed (같은 입력 → "Self")
        # need_weights=False: 어텐션 가중치를 반환하지 않음 (속도 최적화)
        # _: 어텐션 가중치 (여기서는 사용하지 않으므로 _ 로 받음)
        attn_out, _ = self.mha(
            query    = normed,
            key      = normed,
            value    = normed,
            attn_mask = mask    # None이면 마스크 없음 (Encoder 기본값)
        )
        # attn_out shape: (batch, seq_len, d_model)

        # Dropout + 잔차 연결 (Residual Connection)
        x = x + self.dropout(attn_out)
        # x shape: (batch, seq_len, d_model) — 여전히 동일!

        # ════════════════════════════════════════════════
        # 서브레이어 2: Feed-Forward Network
        # ════════════════════════════════════════════════

        # Pre-LN + FFN + Dropout + 잔차 연결
        # 한 줄로 쓰면: x = x + dropout(FFN(LayerNorm(x)))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        # x shape: (batch, seq_len, d_model) — 변화 없음!

        return x


print("TransformerEncoderBlock 클래스 정의 완료!")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 6-A: 기본 동작 테스트
# ──────────────────────────────────────────────────────────────────

# 블록 생성
block = TransformerEncoderBlock(d_model=512, num_heads=8, d_ff=2048, dropout=0.1)

# 입력: 2개 문장, 각 10 토큰, 각 토큰 512차원
x = torch.randn(batch_size, seq_len, d_model)

# 추론 모드 (dropout 비활성화)
block.eval()
with torch.no_grad():
    out = block(x)

print("=" * 55)
print(" Transformer Encoder Block 기본 테스트")
print("=" * 55)
print(f"입력  x.shape   = {x.shape}")
print(f"출력  out.shape = {out.shape}")
print()

assert x.shape == out.shape, "입출력 shape이 달라야 하지 않습니다!"
print("✅ 입력과 출력의 shape가 동일합니다!")
print("   이 성질 덕분에 Encoder Block을 원하는 만큼 쌓을 수 있습니다.")
print()

# ── 파라미터 수 분석 ──
total   = sum(p.numel() for p in block.parameters())
trainable = sum(p.numel() for p in block.parameters() if p.requires_grad)

print("파라미터 수 분석:")
print(f"  전체:       {total:>12,}개")
print(f"  학습 가능:  {trainable:>12,}개")
print()
print("구성 요소별:")
for name, module in block.named_children():
    n = sum(p.numel() for p in module.parameters())
    label = {
        "mha":     "Multi-Head Attention",
        "ffn":     "Feed-Forward Network",
        "ln1":     "LayerNorm 1",
        "ln2":     "LayerNorm 2",
        "dropout": "Dropout (파라미터 없음)",
    }.get(name, name)
    if n > 0:
        print(f"  {label:30s}: {n:>10,}개 ({n/total*100:.1f}%)")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 6-B: 여러 블록 쌓기
# ──────────────────────────────────────────────────────────────────
#
# 입출력 shape가 동일하므로, 블록을 순서대로 쌓으면 됩니다.
# BERT-base:  12개 블록
# BERT-large: 24개 블록
# GPT-3:      96개 블록

class TransformerEncoder(nn.Module):
    """
    N개의 Encoder Block을 쌓은 전체 Transformer Encoder
    """
    def __init__(self, num_layers=6, d_model=512, num_heads=8,
                 d_ff=2048, dropout=0.1):
        super().__init__()
        # nn.ModuleList: 여러 모듈을 리스트로 관리
        # (일반 Python list를 쓰면 파라미터가 등록되지 않습니다!)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)   # num_layers개 블록 생성
        ])
        # 마지막 LayerNorm (출력 정규화)
        self.ln_final = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # x: (batch, seq_len, d_model)
        # 각 블록을 순서대로 통과
        for layer in self.layers:
            x = layer(x, mask)   # 출력이 다음 블록의 입력이 됩니다
        return self.ln_final(x)  # 마지막 정규화


# ── 비교 테스트 ──
print("블록 수에 따른 파라미터 비교")
print("=" * 55)

configs = [
    (1,  256, 4,  1024, "테스트용 (소형)"),
    (6,  512, 8,  2048, "원래 논문 (Transformer-base)"),
    (12, 768, 12, 3072, "BERT-base"),
    (24, 1024,16, 4096, "BERT-large"),
]

x_test = torch.randn(1, 10, 512)
for num_layers, d_model_, num_heads_, d_ff_, label in configs:
    enc = TransformerEncoder(num_layers, d_model_, num_heads_, d_ff_)
    n   = sum(p.numel() for p in enc.parameters())
    print(f"{label}")
    print(f"  {num_layers}개 블록, d_model={d_model_}, heads={num_heads_}, "
          f"d_ff={d_ff_}")
    print(f"  파라미터 수: {n:>12,}개  ({n/1e6:.1f}M)")
    print()

## Step 7: 실험 — 어텐션 마스크 (Attention Mask)

마스크(mask)는 **특정 위치의 어텐션을 차단**하는 장치입니다.

### 언제 필요한가?

1. **패딩 마스크**: 짧은 문장을 같은 길이로 맞추기 위해 추가한 `[PAD]` 토큰을 무시
2. **인과적 마스크 (Causal Mask)**: Decoder에서 미래 토큰을 보지 못하게 차단  
   → 예: "나는 오늘 [?] 갔다"를 예측할 때, 정답 "학교에"를 미리 보면 안 됩니다!

### 인과적 마스크 예시 (seq_len=4)

```
         나는  오늘  학교에  갔다
나는  [  ✅    ❌    ❌      ❌  ]  ← "나는"은 자기 자신만 참조
오늘  [  ✅    ✅    ❌      ❌  ]  ← "오늘"은 "나는", "오늘"만 참조
학교에[  ✅    ✅    ✅      ❌  ]  ← 과거+현재만 참조
갔다  [  ✅    ✅    ✅      ✅  ]  ← 모든 토큰 참조 가능
```

❌ 위치에는 `-inf`를 넣어 softmax 후 0에 가깝게 만듭니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 7-A: 인과적 마스크 생성
# ──────────────────────────────────────────────────────────────────

def create_causal_mask(seq_len):
    """
    인과적(Causal) 마스크 생성
    미래 위치(오른쪽 상단 삼각형)에 -inf를 넣어 어텐션을 0으로 만듭니다.

    Args:
        seq_len (int): 시퀀스 길이
    Returns:
        Tensor: shape (seq_len, seq_len)
                현재/과거 위치: 0 (어텐션 허용)
                미래 위치:     -inf (어텐션 차단)
    """
    # torch.triu: 상삼각 행렬 (diagonal=1 → 대각선 위쪽)
    # diagonal=0: 대각선 포함 위쪽, diagonal=1: 대각선 제외 위쪽
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
    # 1인 위치(미래)를 -inf로, 0인 위치(현재/과거)를 0.0으로
    mask = mask.masked_fill(mask == 1, float('-inf'))
    return mask


# ── 마스크 확인 ──
tokens = ["나는", "오늘", "학교에", "갔다"]
seq_n  = len(tokens)
mask_c = create_causal_mask(seq_n)

print("인과적 마스크 (숫자 표현):")
print(mask_c)
print()
print("해석: 0 = 어텐션 허용 (볼 수 있음), -inf = 차단 (볼 수 없음)")
print("      -inf는 softmax 후 e^(-inf) ≈ 0 이 됩니다.")
print()

# ── 마스크 전후 어텐션 가중치 비교 시각화 ──
d_small = 32
block_small = TransformerEncoderBlock(d_model=d_small, num_heads=4, d_ff=64, dropout=0.0)
x_small = torch.randn(1, seq_n, d_small)

# 어텐션 가중치를 추출하기 위해 직접 mha를 호출
block_small.eval()
with torch.no_grad():
    normed_s = block_small.ln1(x_small)

    # 마스크 없음
    _, w_no_mask = block_small.mha(
        normed_s, normed_s, normed_s, need_weights=True)

    # 인과적 마스크 적용
    _, w_masked = block_small.mha(
        normed_s, normed_s, normed_s,
        attn_mask=mask_c, need_weights=True)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, w, title in zip(
    axes,
    [w_no_mask[0].detach().numpy(), w_masked[0].detach().numpy()],
    ["마스크 없음 (Encoder 기본)", "인과적 마스크 적용 (Decoder 방식)"]
):
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(seq_n))
    ax.set_yticks(range(seq_n))
    ax.set_xticklabels(tokens, fontsize=12)
    ax.set_yticklabels(tokens, fontsize=12)
    ax.set_xlabel("Key (참조 대상)", fontsize=11)
    ax.set_ylabel("Query (관점)",    fontsize=11)
    ax.set_title(title, fontsize=12)
    for i in range(seq_n):
        for j in range(seq_n):
            c = 'white' if w[i, j] > 0.6 else 'black'
            ax.text(j, i, f"{w[i,j]:.2f}", ha='center', va='center',
                    fontsize=10, color=c)

plt.suptitle("어텐션 가중치 비교: 마스크 유무", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print()
print("왼쪽: 모든 토큰이 서로를 참조할 수 있습니다 (Encoder 기본)")
print("오른쪽: 미래 토큰(오른쪽 상단)의 어텐션이 0이 되었습니다 (인과적 마스크)")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 7-B: 하이퍼파라미터 변경 실험
# ──────────────────────────────────────────────────────────────────
#
# 직접 값을 바꿔 보며 파라미터 수와 출력 shape를 확인해 보세요!

print("하이퍼파라미터 변경 실험")
print("=" * 60)

experiment_configs = [
    # (d_model, num_heads, d_ff, 설명)
    (128,  2,  512,  "초소형 (d_model=128)"),
    (256,  4,  1024, "소형   (d_model=256)"),
    (512,  8,  2048, "기본   (원래 논문, d_model=512)"),
    (768, 12,  3072, "중형   (BERT-base, d_model=768)"),
]

for d_m, n_h, d_f, label in experiment_configs:
    b = TransformerEncoderBlock(d_model=d_m, num_heads=n_h, d_ff=d_f)
    n_params = sum(p.numel() for p in b.parameters())

    x_e = torch.randn(1, 10, d_m)  # (batch=1, seq=10, d_model)
    b.eval()
    with torch.no_grad():
        out_e = b(x_e)

    print(f"{label}")
    print(f"  설정       : d_model={d_m}, heads={n_h}, d_ff={d_f}")
    print(f"  파라미터 수: {n_params:>10,}개  ({n_params/1e6:.2f}M)")
    print(f"  입출력     : {x_e.shape} → {out_e.shape}")
    print()

print("💡 d_model이 커질수록 파라미터 수가 대략 제곱으로 증가합니다.")
print("   (MHA: 4 × d_model², FFN: 8 × d_model²)")

## 📝 전체 요약

### Transformer Encoder Block 구성 요소 요약

| 구성 요소 | 역할 | 수식 |
|-----------|------|------|
| **Multi-Head Attention** | 토큰 간 관계 학습 (여러 관점) | $\text{softmax}(QK^T/\sqrt{d_k})V$ |
| **Feed-Forward Network** | 위치별 비선형 변환 | $\text{GELU}(xW_1)W_2$ |
| **Layer Normalization** | 값 정규화, 학습 안정화 | $(x-\mu)/\sigma$ |
| **Residual Connection** | 그라디언트 흐름 보장 | $x + \text{SubLayer}(x)$ |

### Shape 흐름 요약

```
입력   : (batch, seq_len, d_model)  예: (2, 10, 512)
  │
  ├─ LayerNorm  → (2, 10, 512)
  ├─ MHA        → (2, 10, 512)  [내부: (2,8,10,64) → (2,10,512)]
  ├─ Residual   → (2, 10, 512)
  │
  ├─ LayerNorm  → (2, 10, 512)
  ├─ FFN        → (2, 10, 512)  [내부: → 2048 → 512]
  ├─ Residual   → (2, 10, 512)
  │
출력   : (2, 10, 512)  ← 입력과 동일!
```

### 다음 단계

- **여러 블록 쌓기** → 완전한 Transformer Encoder (BERT 등)
- **Decoder Block** 추가 → Cross-Attention이 더해짐
- **Positional Encoding** → 순서 정보 추가 (토큰의 위치를 모델에게 알려줌)
- **사전 학습(Pre-training)** → BERT의 MLM, GPT의 CLM 목적 함수

---

> 💡 **학습 팁:** 숫자를 외우기보다 **shape 변화**와 **정보가 이동하는 방향**을 그림으로 그려보세요!  
> 각 셀의 `verbose=True` 옵션이나 `print(x.shape)`를 활용해  
> 직접 텐서를 추적하는 것이 가장 효과적인 학습 방법입니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 마무리: 전체 흐름 한 번에 실행해보기
# ──────────────────────────────────────────────────────────────────

print("=" * 55)
print(" Transformer Encoder Block 최종 확인")
print("=" * 55)

# 블록 생성
final_block = TransformerEncoderBlock(d_model=512, num_heads=8, d_ff=2048)

# 입력
x_final = torch.randn(2, 10, 512)
print(f"입력:  {x_final.shape}  (2개 문장 × 10 토큰 × 512차원)")

# 순전파
final_block.eval()
with torch.no_grad():
    out_final = final_block(x_final)
print(f"출력:  {out_final.shape}  ← 입력과 동일한 shape ✅")
print()

# 파라미터 요약
total_p = sum(p.numel() for p in final_block.parameters())
print(f"파라미터 수: {total_p:,}개 ({total_p/1e6:.2f}M)")
print()
print("구성 요소별:")
part_labels = {"mha": "Multi-Head Attention", "ffn": "FFN",
               "ln1": "LayerNorm 1", "ln2": "LayerNorm 2"}
for name, module in final_block.named_children():
    n = sum(p.numel() for p in module.parameters())
    if n > 0:
        label = part_labels.get(name, name)
        bar   = "█" * int(n / total_p * 40)
        print(f"  {label:22s}: {n:>8,}개  {bar}")
print()
print("=" * 55)
print(" 실습 완료! 🎉")
print("=" * 55)
print()
print("다음 공부 방향:")
print("  1. Positional Encoding을 추가해 단어 순서 정보를 넣어보세요")
print("  2. Decoder Block을 만들어 번역 모델을 완성해보세요")
print("  3. HuggingFace의 BERT 가중치를 불러와 직접 비교해보세요")